## 1. 🔗 Connexion Google Drive

Montage du Google Drive pour accéder aux données préprocessées et sauvegarder le modèle.  
Le dossier `preprocessed_full` devrait contenir les volumes NIfTI déjà réduits à `128×128×80`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. ⚙️ Installations & Variables Globales

Installation des dépendances MONAI et nibabel.  
Définition de tous les hyperparamètres et chemins utilisés dans le notebook.

| Paramètre | Valeur | Raison |
|-----------|--------|--------|
| `pixdim` | (1.5, 1.5, 1.0) | Isotropie axiale, résolution Z préservée |
| `a_min/a_max` | -200 / 200 HU | Fenêtre abdominale standard CT |
| `spatial_size` | 128×128×80 | Compromis mémoire / résolution |
| `seed` | 42 | Reproductibilité complète |

In [ ]:
!pip install monai nibabel -q

import os, tarfile, torch, numpy as np, random, psutil
from monai.utils import set_determinism

seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
set_determinism(seed=seed)

raw_dir      = '/content/raw_dataset'
drive_dir    = '/content/drive/MyDrive/Liver_Segmentation'
preprocessed = f'{drive_dir}/preprocessed_full'
model_dir    = f'{drive_dir}/results_full'

device       = torch.device('cuda')
pixdim       = (1.5, 1.5, 1.0)
a_min, a_max = -200, 200
spatial_size = [128, 128, 80]

for split in ['Train/images', 'Train/labels', 'Test/images', 'Test/labels']:
    os.makedirs(f'{preprocessed}/{split}', exist_ok=True)
os.makedirs(model_dir, exist_ok=True)
os.makedirs(raw_dir,   exist_ok=True)

print('✅ Variables initialisées !')
print(f'   Drive      : {drive_dir}')
print(f'   Préprocessé: {preprocessed}')
print(f'   Modèle     : {model_dir}')
print(f'   GPU        : {torch.cuda.get_device_name(0)}')

## 3. ⬇️ Téléchargement & Extraction du Dataset

Téléchargement du dataset **Medical Segmentation Decathlon — Task03 Liver** (~27GB) depuis le serveur AWS officiel.  
La cellule est idempotente : si le dataset est déjà présent dans `/content/`, elle est ignorée.

> ⚠️ `/content/` est temporaire — son contenu disparaît à la fin de chaque session Colab.

In [ ]:
tar_local = f'{raw_dir}/Task03_Liver.tar'

if not os.path.exists(f'{raw_dir}/Task03_Liver'):
    if not os.path.exists(tar_local):
        print('⬇️ Téléchargement (~27GB)...')
        !wget -q --show-progress \
          "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task03_Liver.tar" \
          -O {tar_local}
    print('📦 Extraction...')
    with tarfile.open(tar_local, 'r') as tar:
        tar.extractall(raw_dir)
    print('✅ Dataset extrait !')
else:
    print('✅ Dataset déjà présent dans /content/')

## 4. 🔍 Validation des Fichiers

Filtrage des fichiers valides (exclusion des métadonnées Mac `._*`) et détection des fichiers NIfTI corrompus.  
Le fichier `liver_67.nii.gz` (index 95) est systématiquement corrompu dans ce dataset — il est retiré automatiquement.

In [ ]:
import nibabel as nib

base      = f'{raw_dir}/Task03_Liver'
images_tr = sorted([f for f in os.listdir(f'{base}/imagesTr')
                    if f.endswith('.nii.gz') and not f.startswith('._')])
labels_tr = sorted([f for f in os.listdir(f'{base}/labelsTr')
                    if f.endswith('.nii.gz') and not f.startswith('._')])

all_images = [f'{base}/imagesTr/{f}' for f in images_tr]
all_labels = [f'{base}/labelsTr/{f}' for f in labels_tr]

print(f'🔍 Vérification des fichiers ({len(all_images)} patients)...')
corrompus = []
for i, (img_path, lbl_path) in enumerate(zip(all_images, all_labels)):
    try:
        nib.load(img_path).get_fdata()
        nib.load(lbl_path).get_fdata()
        print(f'  ✅ Patient {i:3d}', end='\r')
    except Exception as e:
        print(f'  ❌ Patient {i:3d} CORROMPU : {img_path.split("/")[-1]}')
        corrompus.append(i)

all_images = [img for i, img in enumerate(all_images) if i not in corrompus]
all_labels = [lbl for i, lbl in enumerate(all_labels) if i not in corrompus]
print(f'\n✅ Patients valides : {len(all_images)} (corrompus retirés : {corrompus})')

## 5. 🔄 Preprocessing & Sauvegarde sur Drive

**À exécuter une seule fois.** La cellule est protégée par une guard clause : si les fichiers existent déjà sur Drive, elle s'arrête immédiatement.

Pipeline appliqué sur chaque volume :
1. `LoadImaged` → chargement NIfTI
2. `Spacingd` → rééchantillonnage physique uniforme (bilinear image / nearest label)
3. `Orientationd` → alignement RAS standard
4. `ScaleIntensityRanged` → normalisation HU [-200, +200] → [0.0, 1.0]
5. `CropForegroundd` → suppression du fond noir
6. `Resized` → redimensionnement fixe 128×128×80
7. `Lambdad` → arrondi du label pour garantir les classes entières {0, 1, 2}

Les volumes préprocessés sont sauvegardés en `.nii` sur Google Drive — les sessions futures chargent directement depuis Drive sans retraitement.

In [ ]:
train_done = len(os.listdir(f'{preprocessed}/Train/images')) > 0
test_done  = len(os.listdir(f'{preprocessed}/Test/images'))  > 0

if train_done and test_done:
    print('✅ Preprocessing déjà terminé — cellule ignorée.')
    print(f'   Train : {len(os.listdir(f"{preprocessed}/Train/images"))} fichiers')
    print(f'   Test  : {len(os.listdir(f"{preprocessed}/Test/images"))} fichiers')
else:
    from monai.transforms import (
        Compose, LoadImaged, EnsureChannelFirstd, Spacingd,
        Orientationd, ScaleIntensityRanged, CropForegroundd, Resized, Lambdad
    )
    import nibabel as nib_save

    preprocess = Compose([
        LoadImaged(keys=['image', 'label']),
        EnsureChannelFirstd(keys=['image', 'label']),
        Spacingd(keys=['image', 'label'], pixdim=pixdim, mode=('bilinear', 'nearest')),
        Orientationd(keys=['image', 'label'], axcodes='RAS'),
        ScaleIntensityRanged(keys=['image'], a_min=a_min, a_max=a_max,
                             b_min=0.0, b_max=1.0, clip=True),
        CropForegroundd(keys=['image', 'label'], source_key='image'),
        Resized(keys=['image', 'label'], spatial_size=spatial_size,
                mode=('trilinear', 'nearest')),
        Lambdad(keys=['label'], func=lambda x: x.round().long().float()),
    ])

    split_idx   = int(len(all_images) * 0.9)
    train_pairs = list(zip(all_images[:split_idx], all_labels[:split_idx]))
    test_pairs  = list(zip(all_images[split_idx:], all_labels[split_idx:]))
    print(f'Train : {len(train_pairs)} | Test : {len(test_pairs)}')
    print('💾 Préprocessing + sauvegarde sur Drive...')

    for subset, pairs in [('Train', train_pairs), ('Test', test_pairs)]:
        for i, (img_path, lbl_path) in enumerate(pairs):
            fname   = img_path.split('/')[-1].replace('.nii.gz', '.nii')
            out_img = f'{preprocessed}/{subset}/images/{fname}'
            out_lbl = f'{preprocessed}/{subset}/labels/{fname}'

            if os.path.exists(out_img):
                print(f'  ⏭️ {subset} {i+1}/{len(pairs)} déjà fait', end='\r')
                continue

            sample = preprocess({'image': img_path, 'label': lbl_path})
            nib_save.save(nib_save.Nifti1Image(
                sample['image'][0].numpy().astype(np.float32), np.eye(4)), out_img)
            nib_save.save(nib_save.Nifti1Image(
                sample['label'][0].numpy().astype(np.int8), np.eye(4)), out_lbl)
            print(f'  ✅ {subset} {i+1}/{len(pairs)} → {fname}', end='\r')

    print('\n✅ Préprocessing terminé — fichiers sur Drive !')

## 6. 📦 DataLoaders & Transforms

Chargement des volumes préprocessés depuis Drive et construction des pipelines d'augmentation.

**Augmentations appliquées uniquement sur le train :**

| Transform | Rôle | Paramètres |
|-----------|------|------------|
| `RandFlipd` (×3) | Retournement sur les 3 axes | prob=0.5 |
| `RandRotate90d` | Rotation 90°/180°/270° | prob=0.5 |
| `RandZoomd` | Zoom aléatoire ±15% | prob=0.4 |
| `RandScaleIntensityd` | Variation d'intensité ±15% | prob=0.5 |
| `RandShiftIntensityd` | Décalage d'intensité ±10% | prob=0.5 |
| `RandGaussianNoised` | Bruit gaussien | prob=0.3 |
| `RandGaussianSmoothd` | Lissage gaussien | prob=0.2 |

> Les 4 dernières transforms simulent la variabilité inter-scanners — crucial avec seulement 131 patients.

In [ ]:
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, EnsureTyped,
    RandFlipd, RandRotate90d, RandGaussianNoised, ToTensord,
    RandScaleIntensityd, RandShiftIntensityd, RandZoomd,
    RandGaussianSmoothd
)
from monai.data import Dataset, DataLoader
from glob import glob

train_images = sorted(glob(f'{preprocessed}/Train/images/*.nii'))
train_labels = sorted(glob(f'{preprocessed}/Train/labels/*.nii'))
test_images  = sorted(glob(f'{preprocessed}/Test/images/*.nii'))
test_labels  = sorted(glob(f'{preprocessed}/Test/labels/*.nii'))

assert len(train_images) > 0, '❌ Aucun fichier Train trouvé ! Vérifiez le chemin Drive.'
assert len(test_images)  > 0, '❌ Aucun fichier Test trouvé ! Vérifiez le chemin Drive.'

train_files = [{'image': img, 'label': lbl} for img, lbl in zip(train_images, train_labels)]
test_files  = [{'image': img, 'label': lbl} for img, lbl in zip(test_images, test_labels)]
print(f'✅ Train : {len(train_files)} | Test : {len(test_files)}')

train_transforms = Compose([
    LoadImaged(keys=['image', 'label']),
    EnsureChannelFirstd(keys=['image', 'label']),
    EnsureTyped(keys=['image'], dtype=torch.float32),
    EnsureTyped(keys=['label'], dtype=torch.long),
    RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=0),
    RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=1),
    RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=2),
    RandRotate90d(keys=['image', 'label'], prob=0.5, max_k=3),
    RandZoomd(keys=['image', 'label'], prob=0.4, min_zoom=0.85, max_zoom=1.15,
              mode=('trilinear', 'nearest')),
    RandScaleIntensityd(keys=['image'], prob=0.5, factors=0.15),
    RandShiftIntensityd(keys=['image'], prob=0.5, offsets=0.10),
    RandGaussianNoised(keys=['image'], prob=0.3, mean=0.0, std=0.1),
    RandGaussianSmoothd(keys=['image'], prob=0.2,
                        sigma_x=(0.5, 1.0), sigma_y=(0.5, 1.0), sigma_z=(0.5, 1.0)),
    ToTensord(keys=['image', 'label']),
])

test_transforms = Compose([
    LoadImaged(keys=['image', 'label']),
    EnsureChannelFirstd(keys=['image', 'label']),
    EnsureTyped(keys=['image'], dtype=torch.float32),
    EnsureTyped(keys=['label'], dtype=torch.long),
    ToTensord(keys=['image', 'label']),
])

train_ds = Dataset(data=train_files, transform=train_transforms)
test_ds  = Dataset(data=test_files,  transform=test_transforms)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

print(f'✅ Train loader : {len(train_loader)} batches')
print(f'✅ Test loader  : {len(test_loader)} batches')

## 7. 🧠 Modèle, Loss & Optimizer

**Architecture :** 3D U-Net MONAI avec unités résiduelles  
**3 classes de sortie :** background (0) · foie (1) · tumeur (2)

| Paramètre | Valeur | Raison |
|-----------|--------|--------|
| `out_channels` | 3 | Segmentation multi-classe foie + tumeur |
| `Loss` | DiceCELoss (softmax) | Combine Dice + Cross-Entropy pour mieux détecter les petites tumeurs |
| `Optimizer` | Adam lr=1e-4 | Standard segmentation médicale |
| `Scheduler` | ReduceLROnPlateau | patience=25 pour résister au bruit du petit test set (14 patients) |

> **Pourquoi DiceCELoss plutôt que DiceLoss ?**  
> La tumeur est une petite structure rare dans le volume. La Cross-Entropy pénalise mieux les voxels mal classifiés que le Dice seul.

In [ ]:
from monai.networks.nets import UNet
from monai.networks.layers import Norm
from monai.losses import DiceCELoss

model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=3,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    norm=Norm.BATCH,
).to(device)

loss_function = DiceCELoss(to_onehot_y=True, softmax=True, squared_pred=True)

optimizer = torch.optim.Adam(
    model.parameters(), lr=1e-4, weight_decay=1e-4, amsgrad=True)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=25, min_lr=1e-6)

print(f'✅ Modèle sur     : {next(model.parameters()).device}')
print(f'✅ Paramètres     : {sum(p.numel() for p in model.parameters()):,}')
print(f'✅ out_channels   : 3 (background / foie / tumeur)')
print(f'✅ Loss           : DiceCELoss softmax multiclasse')
print(f'✅ Scheduler      : patience=25, min_lr=1e-6')

model.eval()
with torch.no_grad():
    test_batch = next(iter(train_loader))
    test_img   = test_batch['image'].to(device)
    test_lbl   = test_batch['label'].to(device)
    test_out   = model(test_img)
    test_loss  = loss_function(test_out, test_lbl)

print(f'\n🔍 Forward pass : input {list(test_img.shape)} → output {list(test_out.shape)}')
print(f'   Loss initiale : {test_loss.item():.4f}')
print(f'   VRAM utilisée : {torch.cuda.memory_allocated()/1e9:.2f}GB')

## 8. 🚀 Boucle d'Entraînement

**200 epochs** sur GPU Tesla T4.

**Deux métriques distinctes :**
- **Train** : `dice_metric()` — Dice global avec background (indicateur de convergence rapide)
- **Test** : `DiceMetric(include_background=False)` — Dice par classe sans background (vraie mesure foie + tumeur)

Le meilleur modèle est sauvegardé automatiquement sur Drive à chaque amélioration du Dice moyen (foie + tumeur).

In [ ]:
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete

def dice_metric(predicted, target):
    dice_value = DiceLoss(to_onehot_y=True, softmax=True, squared_pred=True)
    return 1 - dice_value(predicted, target).item()

dice_metric_fn = DiceMetric(include_background=False, reduction='mean_batch')
post_pred      = AsDiscrete(argmax=True, to_onehot=3)
post_label     = AsDiscrete(to_onehot=3)

max_epochs        = 200
best_metric       = -1
best_metric_epoch = -1
save_loss_train, save_loss_test     = [], []
save_metric_train, save_metric_test = [], []

for epoch in range(max_epochs):
    print(f"{'─'*10} epoch {epoch+1}/{max_epochs}")
    model.train()
    train_epoch_loss, train_step, epoch_metric_train = 0, 0, 0

    for batch_data in train_loader:
        train_step += 1
        print(f'    batch {train_step}/{len(train_loader)}', end='\r')
        volume = batch_data['image'].to(device)
        label  = batch_data['label'].to(device)
        optimizer.zero_grad()
        outputs    = model(volume)
        train_loss = loss_function(outputs, label)
        train_loss.backward()
        optimizer.step()
        train_epoch_loss   += train_loss.item()
        epoch_metric_train += dice_metric(outputs, label)

    train_epoch_loss   /= train_step
    epoch_metric_train /= train_step
    save_loss_train.append(train_epoch_loss)
    save_metric_train.append(epoch_metric_train)
    np.save(os.path.join(model_dir, 'loss_train.npy'),   save_loss_train)
    np.save(os.path.join(model_dir, 'metric_train.npy'), save_metric_train)
    print(f'  → Epoch loss: {train_epoch_loss:.4f} | Epoch dice: {epoch_metric_train:.4f}')

    model.eval()
    with torch.no_grad():
        test_epoch_loss, test_step = 0, 0

        for test_data in test_loader:
            test_step += 1
            print(f'    test {test_step}/{len(test_loader)}', end='\r')
            test_volume  = test_data['image'].to(device)
            test_label   = test_data['label'].to(device)
            test_outputs = model(test_volume)
            test_loss    = loss_function(test_outputs, test_label)
            test_epoch_loss += test_loss.item()
            dice_metric_fn(
                [post_pred(test_outputs[0])],
                [post_label(test_label[0])]
            )

        test_epoch_loss   /= test_step
        metric_vals        = dice_metric_fn.aggregate()
        dice_metric_fn.reset()
        epoch_metric_test  = metric_vals.mean().item()

        save_loss_test.append(test_epoch_loss)
        save_metric_test.append(epoch_metric_test)
        np.save(os.path.join(model_dir, 'loss_test.npy'),   save_loss_test)
        np.save(os.path.join(model_dir, 'metric_test.npy'), save_metric_test)

        scheduler.step(epoch_metric_test)
        current_lr = optimizer.param_groups[0]['lr']
        ram        = psutil.virtual_memory()
        vram       = torch.cuda.memory_allocated()/1e9

        print(f'  → Test loss: {test_epoch_loss:.4f} | lr: {current_lr:.2e}')
        print(f'  🫁 Foie: {metric_vals[0]:.4f} | 🔴 Tumeur: {metric_vals[1]:.4f} | 📊 Moy: {epoch_metric_test:.4f}')
        print(f'  🔍 RAM: {ram.used/1e9:.1f}GB/{ram.total/1e9:.1f}GB | VRAM: {vram:.2f}GB')

        if epoch_metric_test > best_metric:
            best_metric       = epoch_metric_test
            best_metric_epoch = epoch + 1
            torch.save(model.state_dict(), os.path.join(model_dir, 'best_metric_model.pth'))
            print(f'  ✅ Nouveau meilleur modèle ! Moy: {best_metric:.4f} '
                  f'(Foie: {metric_vals[0]:.4f} | Tumeur: {metric_vals[1]:.4f})')

        print(f'  Best dice: {best_metric:.4f} at epoch {best_metric_epoch}')

print(f'\n🏆 Entraînement terminé — best dice: {best_metric:.4f} at epoch {best_metric_epoch}')

## 9. 🔍 Debug Ressources (RAM / VRAM)

Cellule utilitaire pour vérifier l'état des ressources à tout moment.  
À exécuter si des crashes mémoire sont suspectés.

In [ ]:
import psutil, torch

ram = psutil.virtual_memory()
print(f'RAM Total   : {ram.total/1e9:.1f}GB')
print(f'RAM Utilisée: {ram.used/1e9:.1f}GB')
print(f'RAM Libre   : {ram.available/1e9:.1f}GB')

print(f'\nVRAM Total   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')
print(f'VRAM Utilisée: {torch.cuda.memory_allocated()/1e9:.1f}GB')
print(f'VRAM Libre   : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.1f}GB')

## 10. 📊 Évaluation Finale du Meilleur Modèle

Chargement du meilleur checkpoint `best_metric_model.pth` et évaluation sur le test set complet.  
La métrique est calculée **sans background** et **par classe** pour avoir une mesure précise par structure.

In [ ]:
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete
import torch

model.load_state_dict(torch.load(f'{model_dir}/best_metric_model.pth'))
model.eval()

post_pred      = AsDiscrete(argmax=True, to_onehot=3)
post_label     = AsDiscrete(to_onehot=3)
dice_metric_fn = DiceMetric(include_background=False, reduction='mean_batch')

with torch.no_grad():
    for test_data in test_loader:
        vol   = test_data['image'].to(device)
        label = test_data['label'].to(device)
        pred  = model(vol)
        dice_metric_fn([post_pred(pred[0])], [post_label(label[0])])

metric = dice_metric_fn.aggregate()
dice_metric_fn.reset()

print(f'🫁 Dice Foie   : {metric[0]:.4f}')
print(f'🔴 Dice Tumeur : {metric[1]:.4f}')
print(f'📊 Dice Moyen  : {((metric[0]+metric[1])/2):.4f}')

## 11. 🖼️ Visualisation — Prédiction vs Ground Truth

Affichage côte à côte de l'image CT, du masque ground truth et de la prédiction du modèle.  
La coupe affichée est celle contenant le plus de voxels foie.

**Légende couleurs :** `0` = background (bleu) · `1` = foie (cyan) · `2` = tumeur (jaune)

In [ ]:
import matplotlib.pyplot as plt
import torch

model.eval()
sample = next(iter(test_loader))
vol    = sample['image'].to(device)
label  = sample['label']

with torch.no_grad():
    pred       = model(vol)
    pred_class = torch.argmax(pred, dim=1)[0].cpu()

vol_np   = vol[0, 0].cpu().numpy()
label_np = label[0, 0].numpy()
pred_np  = pred_class.numpy()

best_z = np.argmax([(label_np[:, :, z] > 0).sum() for z in range(label_np.shape[2])])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(vol_np[:, :, best_z],   cmap='gray')
axes[0].set_title('Image CT')
axes[1].imshow(label_np[:, :, best_z], cmap='jet', vmin=0, vmax=2)
axes[1].set_title('Ground Truth')
axes[2].imshow(pred_np[:, :, best_z],  cmap='jet', vmin=0, vmax=2)
axes[2].set_title('Prédiction')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.savefig(f'{model_dir}/prediction_best_slice.png', dpi=150, bbox_inches='tight')
plt.show()